# DLA fit — JADES-GS-z13-0 (PRISM, source 20128771, z = 12.85)

**Bit-for-bit reproduction of Pollock et al. (2026, A&A, arXiv:2602.11783)** for source `3215_20128771` (JADES-GS-z13-0, z = 12.85).

Pollock+26's recipe (Sect. 3.2):

1. **Identify and fit the rest-frame UV emission lines** (NIV]1486, CIV1548/1550, HeII1640, OIII]1661/66, NIII]1746/52, CIII]1907/09).
2. **Build the joint model (their Eq. 4)**: $F_\lambda = (F_0 \lambda^{\beta_{UV}} + \sum_i G_i)\,e^{-\tau_{DLA}}\,e^{-\tau_{IGM}}$ — the Gaussian lines are added to the intrinsic spectrum *before* DLA + IGM attenuation.
3. **τ_DLA**: exact Voigt–Hjerting (Tepper-García 2006).
4. **τ_IGM**: Miralda-Escudé (1998) / Totani+06, with `z_IGM,upper=z_gal`, `z_IGM,lower=5.3` (Bosman+22).
5. **LSF**: wavelength-dependent NIRSpec PRISM resolution from Jakobsen+22 Fig. 6 × 1.3 (post-launch correction).
6. **Sampler**: dynesty, n_live=500.
7. **Priors**: `log_NHI ∈ U(18,24)`, `x_HI ∈ U(0,1)`, `β_UV ∈ U(−4,0)`, `log_F0 ∈ U(−17,−4)` (in their no-pivot convention; we translate to our `λ_pivot=1500·(1+z)` Å pivot).
8. **Fit range**: 1216–3000 Å rest-frame (Lyα to just below the Balmer break).
9. **Flux units**: F_λ in `erg s⁻¹ cm⁻² Å⁻¹` (cgs).

Pollock+26 quote, for this object: `log(N_HI) = 22.38⁺⁰·¹⁶₋₀·¹⁴`, `x_HI = 0.53⁺⁰·²²₋₀·³²`, `β_UV = −2.62⁺⁰·⁰⁹₋₀·⁰⁸`, `log(F_0) = −9.59⁺⁰·³⁰₋₀·²⁷` (no-pivot, F_λ at λ=1 Å in cgs).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u

import jwspecfit
from jwspecfit.resolution import R_prism

## 1. Load the spectrum and convert to F_λ in cgs

In [ ]:
SPEC_PATH = "../../data/gds-udeep-v4_prism-clear_3215_20128771.spec.fits"
z = 12.85   # JADES-GS-z13-0 (Pollock+26 Fig. 1)

spec = jwspecfit.read_fits(SPEC_PATH, z=z)
print(spec)

wave_A   = spec.wave_um * 1e4
flux_ujy = spec.flux_ujy
err_ujy  = spec.err_ujy

wl_obs = spec.wave_um * u.um
flam = (flux_ujy * u.uJy).to(
    u.erg/u.s/u.cm**2/u.AA, equivalencies=u.spectral_density(wl_obs)
).value
elam = (err_ujy * u.uJy).to(
    u.erg/u.s/u.cm**2/u.AA, equivalencies=u.spectral_density(wl_obs)
).value

## 2. Pre-fit the rest-frame UV emission lines (Pollock+26 Sect. 3.1)

Pollock+26 fit Lyα + UV continuum + emission lines first (their Sect. 3.1) and then *freeze* the line shapes when fitting the DLA in Sect. 3.2.  We do the same — `jwspecfit.fit_lines` on the F_ν spectrum gives us the per-line `(amplitude, centroid, σ)` Gaussians that we will hand to `fit_NHI` as a fixed additive component.

In [ ]:
uv_line_names = [
    "NIV_doublet",
    "CIV_doublet",
    "HEII_1640",
    "OIII_1666",
    "NIII_doublet",
    "CIII]",
]

line_result = jwspecfit.fit_lines(
    spec, z=z, lines=uv_line_names, R=lambda l: R_prism(l), n_boot=200,
)
for name in uv_line_names:
    if name in line_result.lines:
        lr = line_result.lines[name]
        print(f"{name:<14s} flux={lr.flux:.2e}  cen={lr.centroid_A:.1f} A  sig={lr.sigma_A:.2f} A  SNR={lr.snr_int_err:.1f}")

## 3. Build the fixed Gaussian list in F_λ (cgs)

For each detected line (SNR > 2) convert the µJy peak amplitude to F_λ in cgs at the line centroid.  A Gaussian with integrated flux `F_int` (µJy·Å) has peak `A_peak = F_int / (σ √(2π))` (µJy).

In [ ]:
fixed_lines = []
_2pi = np.sqrt(2.0 * np.pi)
for name in uv_line_names:
    if name not in line_result.lines:
        continue
    lr = line_result.lines[name]
    if lr.snr_int_err < 2.0:
        continue
    A_peak_ujy = lr.amplitude / (lr.sigma_A * _2pi)
    A_peak_flam = (A_peak_ujy * u.uJy).to(
        u.erg/u.s/u.cm**2/u.AA,
        equivalencies=u.spectral_density(lr.centroid_A * u.AA),
    ).value
    fixed_lines.append((float(A_peak_flam), float(lr.centroid_A), float(lr.sigma_A)))
    print(f"{name:<14s}  A_peak={A_peak_flam:.2e} cgs  mu={lr.centroid_A:.1f} A  sig={lr.sigma_A:.2f} A")

## 4. Run the DLA + IGM fit — Pollock+26 Eq. 4 verbatim

- **Wavelength array**: observed-frame Å.
- **Flux**: F_λ in cgs.
- **`emission_lines=fixed_lines`**: Gaussians added inside Eq. 4, attenuated by τ_DLA + τ_IGM.
- **`mask_lines=False`**: no need to mask — the lines are inside the model.
- **`R=R_prism`**: wavelength-dependent NIRSpec PRISM curve × 1.3 (Jakobsen+22 + post-launch correction), evaluated per pixel inside the LSF convolution.
- **`igm_z_min=5.3`**: Bosman+22 reionisation end (Pollock+26 Sect. 3.2).
- **`fit_range_A=(1216, 3000)`**: rest-frame, matching Pollock+26.
- **`prior_log_F0`**: Pollock's `(−17, −4)` is for the no-pivot parameterisation `F_λ = F_0 λ^β`.  In our `(λ/λ_pivot)^β` parameterisation with `λ_pivot = 1500·(1+z) Å`, the equivalent prior bounds are `(p − β·log10(λ_pivot))` for both ends of the range; with β anywhere in (−4, 0), `log10(20850) ≈ 4.32`, so we widen the prior to `(−34, −4)` to fully cover Pollock's range and let the auto-derive narrow it.  For practical purposes leaving `prior_log_F0=None` (auto ±3 dex around median continuum) gives an identical posterior.

In [ ]:
result = jwspecfit.fit_NHI(
    wave_A, flam, elam,
    z=z,
    fit_x_HI=True,
    igm_z_min=5.3,
    R=R_prism,                         # wavelength-dependent (Jakobsen+22 x 1.3)
    mask_lines=False,                   # lines are in the model
    mask_lya_emission_width_A=0.0,      # no Lya core to mask
    fit_range_A=(1216.0, 3000.0),       # rest-frame, Pollock+26 Sect. 3.2
    emission_lines=fixed_lines,         # Pollock+26 Eq. 4 Gaussians
    prior_log_NHI=(18.0, 24.0),         # Pollock+26
    prior_x_HI=(0.0, 1.0),              # Pollock+26
    prior_beta_UV=(-4.0, 0.0),          # Pollock+26
    prior_log_F0=None,                  # auto (Pollock prior is no-pivot; equivalent here)
    n_live=500,
    seed=0,
)

print(result.summary())

## 5. Compare to Pollock+26

In [ ]:
ref = {
    "log_NHI": (22.38, 0.16, 0.14),
    "x_HI":    (0.53,  0.22, 0.32),
    "beta_UV": (-2.62, 0.09, 0.08),
    "log_F0":  (-9.59, 0.30, 0.27),
}

lam_pivot = 1500.0 * (1.0 + z)
logF0_pollock = result.log_F0 - result.beta_UV * np.log10(lam_pivot)

ours = {
    "log_NHI": (result.log_NHI, result.log_NHI_err[1], result.log_NHI_err[0]),
    "x_HI":    (result.x_HI,    result.x_HI_err[1],    result.x_HI_err[0]),
    "beta_UV": (result.beta_UV, result.beta_UV_err[1], result.beta_UV_err[0]),
    "log_F0":  (logF0_pollock,  result.log_F0_err[1],  result.log_F0_err[0]),
}

print(f"{'param':<10s} {'this work':<26s} {'Pollock+26':<26s}")
print('-' * 64)
for k in ref:
    o = ours[k]; r = ref[k]
    print(f"{k:<10s} {o[0]:+.2f} (+{o[1]:.2f}, -{o[2]:.2f})     "
          f"{r[0]:+.2f} (+{r[1]:.2f}, -{r[2]:.2f})")

## 6. Visualise the fit

In [ ]:
fig = result.plot(show_residuals=True)
plt.show()

In [ ]:
fig = result.corner(show_titles=True)
plt.show()